# 09 — Round-5 revision analyses (CPU only)

Run this **after** notebook 02 has finished the nine REV4 runs (dropout-matched MobileNetV2 baselines and the FP32
seed-42 ResNet50 runs). If those runs are absent, the corresponding sections are skipped and the rest still runs.

**Attach as inputs:** the consolidated notebook-02 results (all `results/<run>/preds.csv` + `metrics.json`, including
REV4) and the notebook-05 output (`gradcam/gradcam_per_image.csv`).

| Output (in `/kaggle/working/round5/`) | Reviewer item |
|---|---|
| `mobilenet_matched.csv` — MobileNetV2 SE/CBAM vs the dropout-matched baseline, imbalanced and balanced, 5 seeds | MobileNetV2 confound |
| `pooled_matched.csv` — pooled 16 main effects with the matched MobileNetV2 baseline (seed-level t and bootstrap) | MobileNetV2 confound |
| `fp32_seed42.csv` — FP32 vs FP16 seed-42 ResNet50 baseline and SE | +0.060 non-replication |
| `gradcam_within_class.csv` — correct vs incorrect lung energy within each true class | Grad-CAM confound |
| `reliability_bins_resnet50.csv`, `fig_reliability_counts.png` — reliability diagram with bin counts | Fig. 9 |

Zip `/kaggle/working/round5` (last cell) and send the zip back.

In [ ]:
import os, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

RESULT_GLOBS = ['/kaggle/input/**/results/*/metrics.json', '/kaggle/working/results/*/metrics.json']
OUT = '/kaggle/working/round5'; os.makedirs(OUT, exist_ok=True)
N_BOOT, N_BINS = 2000, 15
RNG = np.random.default_rng(12345)
CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']; K = 4
ARCH_ORDER = ['resnet50', 'densenet121', 'mobilenetv2', 'customcnn']
ARCH_LABEL = {'resnet50': 'ResNet50', 'densenet121': 'DenseNet121', 'mobilenetv2': 'MobileNetV2', 'customcnn': 'Custom CNN'}
ATT_LABEL = {'none': 'Baseline', 'se': 'SE', 'cbam': 'CBAM'}

In [ ]:
# ── load runs (as in notebooks 03 and 08) ────────────────────────────────────
metas = {}
for g in RESULT_GLOBS:
    for f in glob.glob(g, recursive=True):
        m = json.load(open(f))
        if m.get('status') != 'done' or (m.get('smoke_test') and os.environ.get('ALLOW_SMOKE') != '1'): continue
        m['dir'] = os.path.dirname(f); metas.setdefault(m['run_id'], m)
print(len(metas), 'finished runs')
RUNS, ref = {}, None
for rid, m in sorted(metas.items()):
    P = pd.read_csv(os.path.join(m['dir'], 'preds.csv'))
    d = P[(P.split == 'test') & (P.ckpt == 'best')].sort_values('idx')
    if ref is None: ref = d.relpath.tolist()
    assert d.relpath.tolist() == ref, f'{rid}: test order differs'
    RUNS[rid] = dict(meta=m, variant=''.join(f'_{k}-{v}' for k, v in sorted(m.get('opts', {}).items())),
                     prob=d[[f'p_{c}' for c in CLASS_NAMES]].values, y=d.y_true.values)
Y = RUNS[next(iter(RUNS))]['y']
CFG = {}
for rid, r in RUNS.items():
    m = r['meta']; CFG.setdefault((m['arch'], m['attention'], m['condition'], r['variant']), {})[m['seed']] = rid
seeds_of = lambda c: sorted(CFG.get(c, {}))

def macro_f1(cm):
    cm = cm.astype(float); tp = np.diagonal(cm, axis1=-2, axis2=-1); fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp
    with np.errstate(divide='ignore', invalid='ignore'):
        p = np.nan_to_num(tp / (tp + fp)); r = np.nan_to_num(tp / (tp + fn)); f = np.nan_to_num(2 * p * r / (p + r))
    return f.mean(-1)
def cm_of(y, p, idx=None):
    if idx is None: return np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    code = y[idx] * K + p[idx]; off = (np.arange(idx.shape[0]) * K * K)[:, None]
    return np.bincount((code + off).ravel(), minlength=idx.shape[0] * K * K).reshape(-1, K, K)
strata = [np.where(Y == c)[0] for c in range(K)]
BOOT = np.concatenate([s[RNG.integers(0, len(s), (N_BOOT, len(s)))] for s in strata], axis=1)   # same draw as notebooks 03/08
F1 = {rid: macro_f1(cm_of(Y, r['prob'].argmax(1))) for rid, r in RUNS.items()}
F1B = {rid: macro_f1(cm_of(Y, r['prob'].argmax(1), BOOT)) for rid, r in RUNS.items()}

def contrast(pairs, seeds):
    """Seed-level t (primary) and paired hierarchical bootstrap for Σ w·mean_s[a_s − b_s]; one shared seed resample per replicate."""
    if not seeds: return None
    pick = RNG.integers(0, len(seeds), (N_BOOT, len(seeds)))
    per_seed = np.zeros(len(seeds)); boot = np.zeros(N_BOOT)
    for a, b, w in pairs:
        da = np.array([F1[CFG[a][s]] - F1[CFG[b][s]] for s in seeds]); per_seed += w * da
        D = np.stack([F1B[CFG[a][s]] - F1B[CFG[b][s]] for s in seeds]); boot += w * D[pick, np.arange(N_BOOT)[:, None]].mean(1)
    n = len(seeds); m = per_seed.mean(); se = per_seed.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
    q, q9 = stats.t.ppf(.975, n - 1), stats.t.ppf(.95, n - 1)
    return dict(delta=m, n_seeds=n, seeds_positive=int((per_seed > 0).sum()), t_lo=m - q * se, t_hi=m + q * se,
                t90_lo=m - q9 * se, t90_hi=m + q9 * se, p_t=stats.ttest_1samp(per_seed, 0).pvalue if n > 1 else np.nan,
                boot_lo=np.percentile(boot, 2.5), boot_hi=np.percentile(boot, 97.5))

In [ ]:
# ── MobileNetV2: attention vs the dropout-matched baseline (REV4) ────────────
rows = []
for cond in ['imbalanced', 'balanced']:
    mb = ('mobilenetv2', 'none', cond, '_head_dropout-0.0')
    if mb not in CFG: print('no matched MobileNetV2 baseline for', cond); continue
    for t in ['se', 'cbam']:
        att = ('mobilenetv2', t, cond, ''); seeds = sorted(set(seeds_of(att)) & set(seeds_of(mb)))
        r = contrast([(att, mb, 1.0)], seeds)
        if r: rows.append(dict(effect=f'{ATT_LABEL[t]} − matched baseline | MobileNetV2 | {cond}', **r))
    dr = ('mobilenetv2', 'none', cond, ''); seeds = sorted(set(seeds_of(dr)) & set(seeds_of(mb)))
    r = contrast([(mb, dr, 1.0)], seeds)
    if r: rows.append(dict(effect=f'matched (no dropout) − dropout baseline | MobileNetV2 | {cond}', **r))
MB = pd.DataFrame(rows); MB.to_csv(f'{OUT}/mobilenet_matched.csv', index=False); print(MB.round(4).to_string())

# pooled 16 main effects with the matched MobileNetV2 baseline (where it has all five seeds in both conditions)
items = []
for cond in ['imbalanced', 'balanced']:
    for a in ARCH_ORDER:
        for t in ['se', 'cbam']:
            base = (a, 'none', cond, '_head_dropout-0.0') if a == 'mobilenetv2' else (a, 'none', cond, '')
            items.append(((a, t, cond, ''), base))
if all(a in CFG and b in CFG and len(seeds_of(b)) == 5 for a, b in items):
    seeds = sorted(set.intersection(*[set(seeds_of(a)) & set(seeds_of(b)) for a, b in items]))
    PM = pd.DataFrame([dict(pool='16 main effects, matched MobileNetV2 baseline', **contrast([(a, b, 1 / 16) for a, b in items], seeds))])
    PM.to_csv(f'{OUT}/pooled_matched.csv', index=False); print(PM.round(4).to_string())
else:
    print('matched MobileNetV2 baseline incomplete — pooled_matched.csv not written')

In [ ]:
# ── FP32 vs FP16 seed-42 ResNet50 runs (REV4) ────────────────────────────────
rows = []
for t in ['none', 'se']:
    fp16 = CFG.get(('resnet50', t, 'imbalanced', ''), {}).get(42); fp32 = CFG.get(('resnet50', t, 'imbalanced', '_amp-False'), {}).get(42)
    if fp16 and fp32:
        rows.append(dict(model=f'ResNet50 {ATT_LABEL[t]}, imbalanced, seed 42', f1_fp16=F1[fp16], f1_fp32=F1[fp32],
                         fp32_minus_fp16=F1[fp32] - F1[fp16],
                         five_seed_fp16_mean=np.mean([F1[r] for r in CFG[('resnet50', t, 'imbalanced', '')].values()]),
                         five_seed_fp16_sd=np.std([F1[r] for r in CFG[('resnet50', t, 'imbalanced', '')].values()], ddof=1)))
FP = pd.DataFrame(rows)
if len(FP) == 2:
    FP.loc[len(FP)] = dict(model='SE − baseline, seed 42', f1_fp16=FP.f1_fp16[1] - FP.f1_fp16[0], f1_fp32=FP.f1_fp32[1] - FP.f1_fp32[0])
FP.to_csv(f'{OUT}/fp32_seed42.csv', index=False); print(FP.round(4).to_string() if len(FP) else 'FP32 runs not found')

In [ ]:
# ── Grad-CAM: correct vs incorrect lung energy WITHIN each true class ────────
gf = glob.glob('/kaggle/input/**/gradcam_per_image.csv', recursive=True) + glob.glob('/kaggle/working/**/gradcam_per_image.csv', recursive=True)
if gf:
    G = pd.read_csv(gf[0]); g = G[G.qc_ok]
    rows = []
    for name in g.model.unique():
        diffs, wts = [], []
        for c in CLASS_NAMES:
            s = g[(g.model == name) & (g.true == c)]
            a, b = s[s.correct].energy_in_lung, s[~s.correct].energy_in_lung
            r = dict(model=name, true=c, n_correct=len(a), n_incorrect=len(b), lung_correct=a.mean(), lung_incorrect=b.mean() if len(b) else np.nan,
                     incorrect_minus_correct=(b.mean() - a.mean()) if len(b) else np.nan)
            if len(b) >= 5 and len(a) >= 5:
                r['mannwhitney_p'] = stats.mannwhitneyu(b, a).pvalue; diffs.append(b.mean() - a.mean()); wts.append(len(b))
            rows.append(r)
        if wts: rows.append(dict(model=name, true='within-class mean (weighted by n incorrect)', incorrect_minus_correct=np.average(diffs, weights=wts)))
        s = g[g.model == name]
        rows.append(dict(model=name, true='pooled across classes (confounded)', n_correct=int(s.correct.sum()), n_incorrect=int((~s.correct).sum()),
                         lung_correct=s[s.correct].energy_in_lung.mean(), lung_incorrect=s[~s.correct].energy_in_lung.mean(),
                         incorrect_minus_correct=s[~s.correct].energy_in_lung.mean() - s[s.correct].energy_in_lung.mean()))
    GW = pd.DataFrame(rows); GW.to_csv(f'{OUT}/gradcam_within_class.csv', index=False); print(GW.round(4).to_string())
else:
    print('gradcam_per_image.csv not found — attach the notebook-05 output')

In [ ]:
# ── reliability diagram with bin counts (ResNet50, pooled over seeds; as notebook 06) ──
rows = []; fig, ax = plt.subplots(2, 2, figsize=(10, 6.4), gridspec_kw=dict(height_ratios=[3, 1.2], hspace=0.08))
cols = {'none': '#4c72b0', 'se': '#dd8452', 'cbam': '#55a868'}
for j, cond in enumerate(['imbalanced', 'balanced']):
    ax[0, j].plot([0, 1], [0, 1], 'k:', lw=1)
    for k_t, t in enumerate(['none', 'se', 'cbam']):
        c = ('resnet50', t, cond, '')
        if c not in CFG: continue
        conf = np.concatenate([RUNS[CFG[c][s]]['prob'].max(1) for s in seeds_of(c)])
        corr = np.concatenate([RUNS[CFG[c][s]]['prob'].argmax(1) == Y for s in seeds_of(c)]).astype(float)
        b = np.minimum((conf * N_BINS).astype(int), N_BINS - 1); xs, ys = [], []
        for k in range(N_BINS):
            n = int((b == k).sum())
            rows.append(dict(condition=cond, attention=ATT_LABEL[t], bin=k, lo=k / N_BINS, hi=(k + 1) / N_BINS, n=n,
                             mean_conf=conf[b == k].mean() if n else np.nan, accuracy=corr[b == k].mean() if n else np.nan, plotted=n >= 10))
            if n >= 10: xs.append(conf[b == k].mean()); ys.append(corr[b == k].mean())
        ax[0, j].plot(xs, ys, 'o-', color=cols[t], ms=4, label=ATT_LABEL[t])
        cnt = np.bincount(b, minlength=N_BINS)
        ax[1, j].bar(np.arange(N_BINS) / N_BINS + (k_t - 1) * 0.02 + 0.5 / N_BINS, cnt, width=0.02, color=cols[t])
    ax[0, j].set(xlim=(0.3, 1.0), ylim=(0.3, 1.0), ylabel='accuracy', title=f'ResNet50 — {cond} training'); ax[0, j].set_xticklabels([])
    ax[0, j].legend(fontsize=8, loc='upper left')
    ax[1, j].set(xlim=(0.3, 1.0), yscale='log', xlabel='confidence (top-label)', ylabel='predictions per bin')
plt.savefig(f'{OUT}/fig_reliability_counts.png', dpi=300, bbox_inches='tight'); plt.close()
pd.DataFrame(rows).to_csv(f'{OUT}/reliability_bins_resnet50.csv', index=False)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/round5', 'zip', OUT); print('wrote /kaggle/working/round5.zip — send this file back')